# Создание приложений для генерации изображений (OpenAI)

Возможности больших языковых моделей не ограничиваются генерацией текста. Вы также можете создавать изображения по текстовым описаниям. Изображения как модальность полезны в таких областях, как медицинские технологии, архитектура, туризм, разработка игр, маркетинг и многое другое. В этом уроке мы работаем с современными моделями **GPT Image** на платформе OpenAI.

## Цели обучения

К концу этого урока вы сможете:

- Объяснить, что такое генерация изображений и где это полезно.
- Понять семью моделей `gpt-image` и чем она отличается от устаревших моделей DALL·E.
- Создать приложение для генерации изображений и редактировать изображения.

## Что такое генерация изображений?

Модели генерации изображений создают изображения по текстовому запросу. Современные модели, такие как `gpt-image`, изучают взаимосвязь между текстом и изображениями во время обучения, а затем поэтапно превращают случайный шум в изображение, соответствующее вашему запросу.

Две известные семьи моделей изображений:

- **`gpt-image` (OpenAI)** — текущее поколение, используемое в этом уроке. Поддерживает генерацию изображений из текста и редактирование изображений (восстановление с маской).
- **Midjourney** — популярная сторонняя модель с собственным сервисом и рабочим процессом на базе Discord.

> Более старые модели OpenAI для работы с изображениями — **DALL·E 2** и **DALL·E 3** — являются устаревшими. DALL·E 3 больше недоступна для новых развертываний, а функции, такие как `create_variation`, были в DALL·E 2. Для новых приложений используйте модели `gpt-image`.

> **Важно:** модели `gpt-image` возвращают сгенерированное изображение в виде строки **base64** (`b64_json`), а не URL. Ваш код декодирует строку base64 в байты и сохраняет — URL изображения для скачивания отсутствует.


## Создание вашего первого приложения для генерации изображений

Итак, что нужно для создания приложения для генерации изображений? Вам потребуются следующие библиотеки:

- **python-dotenv**, настоятельно рекомендуется использовать эту библиотеку, чтобы хранить ваши секреты в файле *.env* отдельно от кода.
- **openai**, эта библиотека используется для взаимодействия с API OpenAI.
- **pillow**, для работы с изображениями в Python.


1. Создайте файл *.env* со следующим содержимым:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Соберите указанные выше библиотеки в файл с именем *requirements.txt* следующим образом:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Далее создайте виртуальное окружение и установите библиотеки:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Для Windows используйте следующие команды для создания и активации вашего виртуального окружения:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Добавьте следующий код в файл с именем *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Давайте объясним этот код:

- Сначала импортируем необходимые библиотеки, включая библиотеку OpenAI, dotenv, модуль base64 и библиотеку Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- После этого создаём клиент, который читает API-ключ из вашего ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Далее генерируем изображение:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` модели возвращают изображение в виде **base64** строки в `data[0].b64_json`. Мы декодируем её в байты и записываем в файл — прямой ссылки для скачивания нет.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Наконец, открываем изображение и используем стандартный просмотрщик для его отображения:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Подробнее о генерации изображения

Рассмотрим параметры `images.generate`:

- **model** — модель изображения для использования (например `gpt-image-2`).
- **prompt** — текстовый запрос для генерации изображения. Здесь это "Кролик на лошади, держащий леденец, на туманном лугу, где растут нарциссы".
- **size** — размер сгенерированного изображения (например `1024x1024`, `1536x1024`, `1024x1536` или `"auto"`).
- **n** — количество сгенерированных изображений. Здесь мы создаём одно.

> Модели изображений не принимают параметр `temperature` — это управление генерацией текста. Чтобы получить разнообразие, вызывайте API снова; чтобы уменьшить разнообразие, сделайте ваш запрос более конкретным.

## Дополнительные возможности генерации изображений

Вы уже видели, как с помощью нескольких строк Python можно сгенерировать изображение. `gpt-image` модели также могут **редактировать** существующее изображение. Предоставив изображение, необязательную **маску** (отмечающую область для изменения) и запрос, вы можете изменить часть изображения — например, добавить шляпу нашему кролику.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Исходное изображение содержит только кролика; итоговое изображение с шляпой на кролике.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
